In [ ]:
# ============================================================
# Statistical significance + effect size tests
# ============================================================

!pip install -q pandas numpy scipy

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import wilcoxon

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 1. Load full results file
# ============================================================

FULL_RESULTS_PATH = "results/full_results.csv"

OUT_DIR = Path("results/statistical_tests")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(FULL_RESULTS_PATH)

print("Rows:", len(df))
print("Methods:", df["method"].nunique())
print(df["method"].unique())

In [ ]:
# ============================================================
# 2. Clean method labels
# ============================================================

METHOD_LABELS = {
    "zero_shot_gpt_no_dataset": "Zero-Shot Opposite-Stance Prompting",
    "few_shot_task_prompting_no_dataset": "Few-Shot Opposite-Stance Prompting",
    "mazare2018_retrieval_only_raw_700k": "Semantic Persona Retrieval (Mazaré et al.)",
    "partner_matters_persona_fusion_response_selection": "Persona Fusion Retrieval (Partner Matters)",
    "coo_style_usefulness_ranking_raw_700k": "Usefulness-Aware Persona Selection (COO)",
    "personadb_offline_hierarchical_refinement": "Hierarchical Persona Refinement (Persona-DB)",
    "ours_stance_nli_gpt_final_repository": "Contradiction-Aware Persona Retrieval (Ours)",
}

OURS_METHOD = "ours_stance_nli_gpt_final_repository"

df["method_label"] = df["method"].map(METHOD_LABELS).fillna(df["method"])

In [ ]:
# ============================================================
# 3. Create stable paired case id
# ============================================================

# Each method was evaluated on the same test cases.
# We identify each case by topic + user opinion + expected stance.

df["case_id"] = (
    df["topic"].astype(str)
    + " || "
    + df["expected_user_stance"].astype(str)
    + " || "
    + df["user_opinion"].astype(str)
)

print("Unique test cases:", df["case_id"].nunique())

In [ ]:
# ============================================================
# 4. Effect size helpers
# ============================================================

def paired_cohens_d(x, y):
    """
    Cohen's dz for paired samples:
    mean difference divided by std of differences.
    Positive value means x > y.
    """
    diff = np.asarray(x, dtype=float) - np.asarray(y, dtype=float)
    diff = diff[~np.isnan(diff)]

    if len(diff) < 2:
        return np.nan

    sd = np.std(diff, ddof=1)
    if sd == 0:
        return np.nan

    return float(np.mean(diff) / sd)


def rank_biserial_from_wilcoxon(x, y):
    """
    Rank-biserial correlation for paired Wilcoxon.
    Range: [-1, 1]
    Positive means x tends to be greater than y.
    """
    diff = np.asarray(x, dtype=float) - np.asarray(y, dtype=float)
    diff = diff[~np.isnan(diff)]
    diff = diff[diff != 0]

    n = len(diff)
    if n == 0:
        return np.nan

    abs_diff = np.abs(diff)
    ranks = pd.Series(abs_diff).rank(method="average").to_numpy()

    w_pos = ranks[diff > 0].sum()
    w_neg = ranks[diff < 0].sum()

    total_rank = n * (n + 1) / 2

    return float((w_pos - w_neg) / total_rank)


def cliffs_delta(x, y):
    """
    Cliff's delta for paired arrays treated as two samples.
    Range: [-1, 1]
    Positive means x > y more often than x < y.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]

    greater = 0
    smaller = 0

    for xi in x:
        greater += np.sum(xi > y)
        smaller += np.sum(xi < y)

    n = len(x) * len(y)
    if n == 0:
        return np.nan

    return float((greater - smaller) / n)


def interpret_effect_size_d(d):
    if pd.isna(d):
        return ""
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    elif ad < 0.5:
        return "small"
    elif ad < 0.8:
        return "medium"
    else:
        return "large"


def interpret_rank_biserial(r):
    if pd.isna(r):
        return ""
    ar = abs(r)
    if ar < 0.1:
        return "negligible"
    elif ar < 0.3:
        return "small"
    elif ar < 0.5:
        return "medium"
    else:
        return "large"

In [ ]:
# ============================================================
# 5. Paired significance test function
# ============================================================

def paired_significance_tests(
    df,
    metric_col,
    ours_method=OURS_METHOD,
    alternative="greater",
):
    """
    Runs paired Wilcoxon signed-rank tests comparing ours vs every baseline.

    alternative="greater" tests whether ours > baseline.
    Use alternative="two-sided" if you prefer a more conservative test.
    """

    pivot = df.pivot_table(
        index="case_id",
        columns="method",
        values=metric_col,
        aggfunc="mean",
    )

    if ours_method not in pivot.columns:
        raise ValueError(f"Could not find ours method: {ours_method}")

    rows = []

    ours_values = pivot[ours_method]

    for baseline in pivot.columns:
        if baseline == ours_method:
            continue

        pair_df = pd.DataFrame({
            "ours": ours_values,
            "baseline": pivot[baseline],
        }).dropna()

        x = pair_df["ours"].to_numpy(dtype=float)
        y = pair_df["baseline"].to_numpy(dtype=float)

        diff = x - y

        # Wilcoxon cannot run if all differences are exactly zero.
        if np.allclose(diff, 0):
            stat = np.nan
            p_value = 1.0
        else:
            stat, p_value = wilcoxon(
                x,
                y,
                alternative=alternative,
                zero_method="wilcox",
            )

        rows.append({
            "metric": metric_col,
            "comparison": f"{METHOD_LABELS.get(ours_method, ours_method)} vs {METHOD_LABELS.get(baseline, baseline)}",
            "ours_method": ours_method,
            "baseline_method": baseline,
            "ours_label": METHOD_LABELS.get(ours_method, ours_method),
            "baseline_label": METHOD_LABELS.get(baseline, baseline),
            "n_pairs": len(pair_df),
            "ours_mean": float(np.mean(x)),
            "baseline_mean": float(np.mean(y)),
            "mean_difference": float(np.mean(diff)),
            "wilcoxon_statistic": float(stat) if not pd.isna(stat) else np.nan,
            "p_value": float(p_value),
            "paired_cohens_d": paired_cohens_d(x, y),
            "rank_biserial_correlation": rank_biserial_from_wilcoxon(x, y),
            "cliffs_delta": cliffs_delta(x, y),
        })

    result_df = pd.DataFrame(rows)

    result_df["cohens_d_interpretation"] = result_df["paired_cohens_d"].apply(
        interpret_effect_size_d
    )

    result_df["rank_biserial_interpretation"] = result_df["rank_biserial_correlation"].apply(
        interpret_rank_biserial
    )

    result_df["significant_p_0_05"] = result_df["p_value"] < 0.05

    return result_df.sort_values("p_value")

In [ ]:
# ============================================================
# 6. Run tests for main metrics
# ============================================================

nli_tests_df = paired_significance_tests(
    df=df,
    metric_col="nli_contradiction",
    alternative="greater",
)

stance_tests_df = paired_significance_tests(
    df=df,
    metric_col="opposite_stance_accuracy",
    alternative="greater",
)

display(nli_tests_df)
display(stance_tests_df)

In [ ]:
# ============================================================
# 7. Optional: two-sided tests
# ============================================================

nli_tests_two_sided_df = paired_significance_tests(
    df=df,
    metric_col="nli_contradiction",
    alternative="two-sided",
)

stance_tests_two_sided_df = paired_significance_tests(
    df=df,
    metric_col="opposite_stance_accuracy",
    alternative="two-sided",
)

display(nli_tests_two_sided_df)
display(stance_tests_two_sided_df)

In [ ]:
# ============================================================
# 8. Save results
# ============================================================

nli_tests_df.to_csv(
    OUT_DIR / "wilcoxon_effect_size_nli_contradiction_greater.csv",
    index=False,
    encoding="utf-8-sig",
)

stance_tests_df.to_csv(
    OUT_DIR / "wilcoxon_effect_size_opposite_stance_accuracy_greater.csv",
    index=False,
    encoding="utf-8-sig",
)

nli_tests_two_sided_df.to_csv(
    OUT_DIR / "wilcoxon_effect_size_nli_contradiction_two_sided.csv",
    index=False,
    encoding="utf-8-sig",
)

stance_tests_two_sided_df.to_csv(
    OUT_DIR / "wilcoxon_effect_size_opposite_stance_accuracy_two_sided.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved statistical test results to:", OUT_DIR)

In [ ]:
# ============================================================
# Create publication-friendly tables
# (remove effect size columns)
# ============================================================

KEEP_COLS = [
    "comparison",
    "n_pairs",
    "ours_mean",
    "baseline_mean",
    "mean_difference",
    "p_value",
    "significant_p_0_05",
]

nli_tests_paper_df = nli_tests_df[KEEP_COLS].copy()
stance_tests_paper_df = stance_tests_df[KEEP_COLS].copy()

display(nli_tests_paper_df)
display(stance_tests_paper_df)

In [ ]:
nli_tests_paper_df.to_csv(
    OUT_DIR / "nli_significance_table_for_paper.csv",
    index=False,
    encoding="utf-8-sig",
)

stance_tests_paper_df.to_csv(
    OUT_DIR / "stance_significance_table_for_paper.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved paper-friendly significance tables.")

In [ ]:
nli_tests_paper_df = nli_tests_df[
    [
        "comparison",
        "mean_difference",
        "p_value",
        "significant_p_0_05",
    ]
].copy()

stance_tests_paper_df = stance_tests_df[
    [
        "comparison",
        "mean_difference",
        "p_value",
        "significant_p_0_05",
    ]
].copy()

In [ ]:
nli_tests_paper_df.to_csv(
    OUT_DIR / "summary_nli_significance_table_for_paper.csv",
    index=False,
    encoding="utf-8-sig",
)

stance_tests_paper_df.to_csv(
    OUT_DIR / "summary_stance_significance_table_for_paper.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved paper-friendly significance tables.")